# 04 · Gradient-based Optimization: Discrete Stepping and Continuous Gradient Descent

In mathematical optimization, derivatives provide the direction of greatest instantaneous change. 
- $\nabla f(x)$ points in the direction of **steepest ascent**.
- $-\nabla f(x)$ points in the direction of **steepest descent**.

In this notebook, we explore:
1. The 1D Sphere function $f(x) = x^2$ and central difference gradient estimation.
2. The discrete **sign-based stepping** algorithm from lecture slides 30–35 ($x_t = x_{t-1} - \text{sign}(f'(x_{t-1})) \cdot \text{step}$).
3. Multi-variable 2D discrete optimization for $f(x, y) = x^2 + y^2$ (slides 36–38).
4. Continuous **Gradient Descent** with learning rates: $\theta_t = \theta_{t-1} - \eta \nabla f(\theta_{t-1})$.
5. Diagnosing convergence behaviors under under-stepping, optimal stepping, and divergent learning rates.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


# 1D Sphere function f(x) = x^2
def sphere_1d(x: float) -> float:
    return x * x


def gradient_1d(f, x: float, epsilon: float = 1e-4) -> float:
    """Compute central difference gradient approximation."""
    return (f(x + epsilon / 2.0) - f(x - epsilon / 2.0)) / epsilon


# Verification
assert np.isclose(gradient_1d(sphere_1d, 5.0), 10.0, atol=1e-3)
assert np.isclose(gradient_1d(sphere_1d, 0.0), 0.0, atol=1e-3)
assert np.isclose(gradient_1d(sphere_1d, -5.0), -10.0, atol=1e-3)
print("gradient(f, 5)  ->", gradient_1d(sphere_1d, 5.0))
print("gradient(f, 0)  ->", gradient_1d(sphere_1d, 0.0))
print("gradient(f, -5) ->", gradient_1d(sphere_1d, -5.0))

## 1. Discrete Sign-based Optimization (Slides 31–35)

When searching over integer lattices or discrete parameters, taking unit steps against the derivative sign minimizes the function:

$$x_t = x_{t-1} - \text{sign}(f'(x_{t-1})) \cdot \text{step}$$

Let us simulate the exact trace from the lecture slides, starting at $x_0 = 70.0$ with $\text{step} = 1.0$.


In [ ]:
# Simulation of slide 33-35
x = 70.0
step = 1.0
trajectory_1d = [x]

for it in range(1, 76):
    dx = gradient_1d(sphere_1d, x)
    sign = np.sign(dx)
    x = x - sign * step
    trajectory_1d.append(x)
    if it in [1, 2, 3, 63, 64, 65, 68, 69, 70]:
        print(f"Step {it:2d}: x = {x:4.1f}, f'(x) = {dx:5.1f}, f(x) = {sphere_1d(x):6.1f}")

assert trajectory_1d[70] == 0.0
print(f"\nReached exact global minimum at iteration 70: x = {x:.1f}")

## 2. 2D Discrete Coordinate Optimization (Slides 36–38)

For a multi-variable function $f(x, y) = x^2 + y^2$, the gradient is:

$$\nabla f(x, y) = \begin{bmatrix} \frac{\partial f}{\partial x} \\ \frac{\partial f}{\partial y} \end{bmatrix} = \begin{bmatrix} 2x \\ 2y \end{bmatrix}$$

Updating each variable along its negative derivative sign:
$$x_t = x_{t-1} - \text{sign}\left(\frac{\partial f}{\partial x}\right)$$
$$y_t = y_{t-1} - \text{sign}\left(\frac{\partial f}{\partial y}\right)$$

Let us trace from $(x_0, y_0) = (3.0, 4.0)$ matching slide 37:
- Step 0: $(3.0, 4.0) \implies \nabla = (6.0, 8.0)$
- Step 1: $(2.0, 3.0) \implies \nabla = (4.0, 6.0)$
- Step 2: $(1.0, 2.0) \implies \nabla = (2.0, 4.0)$
- Step 3: $(0.0, 1.0) \implies \nabla = (0.0, 2.0)$
- Step 4: $(0.0, 0.0)$


In [ ]:
def sphere_2d(x: float, y: float) -> float:
    return x * x + y * y


def grad_2d(f, x: float, y: float, eps: float = 1e-4) -> tuple[float, float]:
    df_dx = (f(x + eps / 2.0, y) - f(x - eps / 2.0, y)) / eps
    df_dy = (f(x, y + eps / 2.0) - f(x, y - eps / 2.0)) / eps
    return df_dx, df_dy


# Trace from (3.0, 4.0)
curr_x, curr_y = 3.0, 4.0
print(f"Step 0: x = {curr_x:.1f}, y = {curr_y:.1f}")

for step_num in range(1, 5):
    gx, gy = grad_2d(sphere_2d, curr_x, curr_y)
    curr_x -= np.sign(gx)
    curr_y -= np.sign(gy)
    print(f"Step {step_num}: x = {curr_x:.1f}, y = {curr_y:.1f}, grad = ({gx:.1f}, {gy:.1f})")

assert curr_x == 0.0 and curr_y == 0.0
print("2D discrete optimization successfully reached origin (0, 0) in 4 steps.")

## 3. Continuous Gradient Descent and Learning Rate Dynamics (Slides 39–41)

In continuous spaces, taking steps proportional to the gradient magnitude accelerates convergence when far from the minimum and slows down as the slope approaches zero:

$$\theta_t = \theta_{t-1} - \eta \nabla f(\theta_{t-1})$$

The choice of learning rate $\eta$ governs convergence:
1. **Too Small ($\eta = 0.05$):** Monotonic but painfully slow convergence; requires hundreds of steps.
2. **Optimal ($\eta = 0.40$):** Geometric convergence to optimum in few iterations.
3. **Too Large / Divergent ($\eta = 1.05$):** For $f(x) = x^2$, the update is $x_t = x_{t-1} - \eta(2x_{t-1}) = (1 - 2\eta)x_{t-1}$. If $|1 - 2\eta| > 1$ (i.e. $\eta > 1.0$), the sequence oscillates with exponentially growing magnitude!


In [ ]:
def run_gradient_descent_1d(x_init: float, lr: float, steps: int = 25) -> list[float]:
    x = x_init
    traj = [x]
    for _ in range(steps):
        grad = 2.0 * x  # exact analytical gradient of x^2
        x = x - lr * grad
        traj.append(x)
    return traj


x_start = 10.0
traj_small = run_gradient_descent_1d(x_start, lr=0.05)
traj_good = run_gradient_descent_1d(x_start, lr=0.40)
traj_large = run_gradient_descent_1d(x_start, lr=1.05)

plt.figure(figsize=(10, 5))
plt.plot(traj_small, label="Small LR (η = 0.05) - Slow", marker="o", markersize=4)
plt.plot(traj_good, label="Good LR (η = 0.40) - Rapid", marker="s", markersize=4)
plt.plot(traj_large, label="Large LR (η = 1.05) - Divergent Oscillations", marker="^", markersize=4)

plt.axhline(0, color="gray", linestyle="--")
plt.xlabel("Iteration", fontsize=12)
plt.ylabel("Parameter Value x", fontsize=12)
plt.ylim(-25, 25)
plt.title("Impact of Learning Rate on Gradient Descent Dynamics", fontsize=14)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 4. 2D Continuous Gradient Descent Trajectory Visualization


In [ ]:
def run_gradient_descent_2d(init_pt: tuple[float, float], lr: float, steps: int = 20):
    x, y = init_pt
    traj = [(x, y)]
    for _ in range(steps):
        gx, gy = 2.0 * x, 2.0 * y
        x -= lr * gx
        y -= lr * gy
        traj.append((x, y))
    return np.array(traj)


traj_2d = run_gradient_descent_2d((8.0, 7.0), lr=0.25, steps=15)

# Contour plot
x_grid = np.linspace(-10, 10, 100)
y_grid = np.linspace(-10, 10, 100)
X, Y = np.meshgrid(x_grid, y_grid)
Z = X**2 + Y**2

plt.figure(figsize=(8, 7))
cp = plt.contour(X, Y, Z, levels=15, cmap="viridis")
plt.clabel(cp, inline=True, fontsize=8)

plt.plot(traj_2d[:, 0], traj_2d[:, 1], "r-o", linewidth=2, label="GD Trajectory (η = 0.25)")
plt.plot(traj_2d[0, 0], traj_2d[0, 1], "g*", markersize=14, label="Start (8, 7)")
plt.plot(0, 0, "rX", markersize=12, label="Global Min (0, 0)")

plt.title("2D Gradient Descent Trajectory on Sphere f(x, y) = x^2 + y^2", fontsize=13)
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.5)
plt.axis("equal")
plt.show()